## A COMPLETED MACHINE LEARNING PROJECT

* THIS PROJECT JOCUSING ON HOW AN END TO END MACHINE LEARNING PROJECT LOOKS LIKE, FROM IDENTIFYING THE BUSINESS PROBLEM, COLLETING DATASET, DOING EXPLORATORY DATA ANALYSIS, FEATURE ENGINEERING TO BUILDING PREDICTIVE MODELLING AND ANSWERING BUSINESS QUESTION. 
* THE DATASET WE USE WAS LOANS_2007 INCLUDING INFORMATION ABOUT THE LOAN STATUS OF THE BORROWERS

IDENTIFYING THE PROBLEMS
* Because we are trying to know whether or not we can predict the probability of the borrower will pay off their loan or not<br>
  so the problem we need to clarify here is: Can we build a machine learning model that can accurately predict if a borrower will pay off their loan on time or not?, and of course my answer is big yes :))

In [14]:
import pandas as pd
import numpy as np
from IPython.display import display
pd.options.display.max_columns = 999

In [15]:
df = pd.read_csv('loans_2007.csv')
df.drop_duplicates()
print(df.shape)
display(df.head(3))

/opt/anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3020: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


(42538, 52)


,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,emp_title,emp_length,home_ownership,annual_inc,verification_status,issue_d,loan_status,pymnt_plan,purpose,title,zip_code,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,out_prncp,out_prncp_inv,total_pymnt,total_pymnt_inv,total_rec_prncp,total_rec_int,total_rec_late_fee,recoveries,collection_recovery_fee,last_pymnt_d,last_pymnt_amnt,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
0,1077501,1296599.0,5000.0,5000.0,4975.0,36 months,10.65%,162.87,B,B2,NaN,10+ years,RENT,24000.0,Verified,Dec-2011,Fully Paid,n,credit_card,Computer,860xx,AZ,27.65,0.0,Jan-1985,1.0,3.0,0.0,13648.0,83.7%,9.0,f,0.0,0.0,5863.155187,5833.84,5000.00,863.16,0.0,0.00,0.00,Jan-2015,171.62,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
1,1077430,1314167.0,2500.0,2500.0,2500.0,60 months,15.27%,59.83,C,C4,Ryder,< 1 year,RENT,30000.0,Source Verified,Dec-2011,Charged Off,n,car,bike,309xx,GA,1.00,0.0,Apr-1999,5.0,3.0,0.0,1687.0,9.4%,4.0,f,0.0,0.0,1008.710000,1008.71,456.46,435.17,0.0,117.08,1.11,Apr-2013,119.66,Sep-2013,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
2,1077175,1313524.0,2400.0,2400.0,2400.0,36 months,15.96%,84.33,C,C5,NaN,10+ years,RENT,12252.0,Not Verified,Dec-2011,Fully Paid,n,small_business,real estate business,606xx,IL,8.72,0.0,Nov-2001,2.0,2.0,0.0,2956.0,98.5%,10.0,f,0.0,0.0,3005.666844,3005.67,2400.00,605.67,0.0,0.00,0.00,Jun-2014,649.91,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0


In [17]:
df.drop(["id", "member_id", "funded_amnt", "funded_amnt_inv", "grade", "sub_grade", "emp_title", "issue_d"], axis=1, inplace = True)
df.shape

(42538, 44)

In [18]:
df.drop(["zip_code", "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv", "total_rec_prncp"], axis=1, inplace=True)
df.shape

(42538, 38)

In [19]:
df.drop(["total_rec_int", "total_rec_late_fee", "recoveries", "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt"], axis=1, inplace=True)
df.shape

(42538, 32)

 Sau khi ta đã lọc bỏ đi bớt những cột không có giá trị, câu hỏi tiếp theo ta cần trả lời đó là : column nào ta sẽ chọn để làm target cho mô hình. Để trả lời câu hỏi này ta hãy quay về vấn đề ta cần giải quyết, đó là xây dựng model dự báo khả năng một người có trả tiền hay không. Như vậy dễ thấy trong tất cả các cột thì loan_status chính là feature phù hợp nhất để chọn, do vậy ta sẽ chọn loan_status làm target predict

In [21]:
df.loan_status.value_counts()

Fully Paid                                             33136
Charged Off                                             5634
Does not meet the credit policy. Status:Fully Paid      1988
Current                                                  961
Does not meet the credit policy. Status:Charged Off      761
Late (31-120 days)                                        24
In Grace Period                                           20
Late (16-30 days)                                          8
Default                                                    3
Name: loan_status, dtype: int64

Điều chúng ta nhận thấy ở các giá trị có trong loan_status thì 2 giá trị đối lập và có ý nghĩa nhất đó là fully_paid(đã trả toàn bộ) và charged off( có nghĩa giống như quỵt nợ, ko thể đòi lại đc), còn những giá trị còn lại vẫn đang trong tình trạng ko rõ ràng có trả hay không, do vậy ta ko quang tâm đến những giá trị này. Điều chúng ta quan tâm ở đây đó là 2 giá trị fully_paid và charge_off. <br>

Ta sẻ chỉ sử dụng những rows có chứa giá trị thuộc về 2 giá trị trên, và do đó ta đưa tập dataset về dạng mà ở đó model chúng ta xây dựng sẽ ở dạng binary classification

In [22]:
df = df[(df.loan_status == 'Fully Paid') | (df.loan_status == 'Charged Off')]
status_replace = {'loan_status': {'Fully Paid':1, 'Charged Off':0}}
df = df.replace(status_replace)
print(df.shape)
df.head(3)

(38770, 32)


,loan_amnt,term,int_rate,installment,emp_length,home_ownership,annual_inc,verification_status,loan_status,pymnt_plan,purpose,title,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,initial_list_status,last_credit_pull_d,collections_12_mths_ex_med,policy_code,application_type,acc_now_delinq,chargeoff_within_12_mths,delinq_amnt,pub_rec_bankruptcies,tax_liens
0,5000.0,36 months,10.65%,162.87,10+ years,RENT,24000.0,Verified,1,n,credit_card,Computer,AZ,27.65,0.0,Jan-1985,1.0,3.0,0.0,13648.0,83.7%,9.0,f,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
1,2500.0,60 months,15.27%,59.83,< 1 year,RENT,30000.0,Source Verified,0,n,car,bike,GA,1.00,0.0,Apr-1999,5.0,3.0,0.0,1687.0,9.4%,4.0,f,Sep-2013,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0
2,2400.0,36 months,15.96%,84.33,10+ years,RENT,12252.0,Not Verified,1,n,small_business,real estate business,IL,8.72,0.0,Nov-2001,2.0,2.0,0.0,2956.0,98.5%,10.0,f,Jun-2016,0.0,1.0,INDIVIDUAL,0.0,0.0,0.0,0.0,0.0


Bây giờ chúng ta sẽ tiến hành lọc thêm 1 số columns không giá trị chẳng hạn như column đó không có lấy quá 1 giá trị khác nhau, Vì sao? tại vì nếu 1 column toàn chứa những giá trị giống hệt nhau thì columns không có ý nghĩa trong xây dựng mô hình

In [24]:
cols = df.columns
drop_cols = []
for col in cols:
    series = df[col].dropna().unique()
    if len(series) == 1:
        drop_cols.append(col)
df = df.drop(drop_cols, axis=1)
print(df.shape)
df.head(3)

(38770, 23)


,loan_amnt,term,int_rate,installment,emp_length,home_ownership,annual_inc,verification_status,loan_status,purpose,title,addr_state,dti,delinq_2yrs,earliest_cr_line,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,last_credit_pull_d,pub_rec_bankruptcies
0,5000.0,36 months,10.65%,162.87,10+ years,RENT,24000.0,Verified,1,credit_card,Computer,AZ,27.65,0.0,Jan-1985,1.0,3.0,0.0,13648.0,83.7%,9.0,Jun-2016,0.0
1,2500.0,60 months,15.27%,59.83,< 1 year,RENT,30000.0,Source Verified,0,car,bike,GA,1.00,0.0,Apr-1999,5.0,3.0,0.0,1687.0,9.4%,4.0,Sep-2013,0.0
2,2400.0,36 months,15.96%,84.33,10+ years,RENT,12252.0,Not Verified,1,small_business,real estate business,IL,8.72,0.0,Nov-2001,2.0,2.0,0.0,2956.0,98.5%,10.0,Jun-2016,0.0


Chúng ta sẽ làm sạch data từ  những missing values

In [27]:
100 * df.isnull().sum() / df.shape[0] 

loan_amnt               0.000000
term                    0.000000
int_rate                0.000000
installment             0.000000
emp_length              2.672169
home_ownership          0.000000
annual_inc              0.000000
verification_status     0.000000
loan_status             0.000000
purpose                 0.000000
title                   0.028372
addr_state              0.000000
dti                     0.000000
delinq_2yrs             0.000000
earliest_cr_line        0.000000
inq_last_6mths          0.000000
open_acc                0.000000
pub_rec                 0.000000
revol_bal               0.000000
revol_util              0.128966
total_acc               0.000000
last_credit_pull_d      0.005159
pub_rec_bankruptcies    1.797782
dtype: float64

In [28]:
df.drop(['emp_length', 'pub_rec_bankruptcies'], axis = 1, inplace=True)
df.dropna(axis = 0, inplace=True)
df.dtypes.value_counts()

object     10
float64    10
int64       1
dtype: int64

In [29]:
df.isnull().sum()

loan_amnt              0
term                   0
int_rate               0
installment            0
home_ownership         0
annual_inc             0
verification_status    0
loan_status            0
purpose                0
title                  0
addr_state             0
dti                    0
delinq_2yrs            0
earliest_cr_line       0
inq_last_6mths         0
open_acc               0
pub_rec                0
revol_bal              0
revol_util             0
total_acc              0
last_credit_pull_d     0
dtype: int64

In [30]:
df_object = df.select_dtypes(include=['object'])
df_object.head(2)

,term,int_rate,home_ownership,verification_status,purpose,title,addr_state,earliest_cr_line,revol_util,last_credit_pull_d
0,36 months,10.65%,RENT,Verified,credit_card,Computer,AZ,Jan-1985,83.7%,Jun-2016
1,60 months,15.27%,RENT,Source Verified,car,bike,GA,Apr-1999,9.4%,Sep-2013


In [38]:
# Ta sẽ xem các giá trị unique có trong các cột giống vs categorical value sau
cols = ['home_ownership', 'verification_status', 'term', 'addr_state']
for col in cols:
    print(f'CÁC GIÁ TRỊ UNIQUE CÓ TRONG {col} là: ', df[col].value_counts())

CÁC GIÁ TRỊ UNIQUE CÓ TRONG home_ownership là:  RENT        18513
MORTGAGE    17111
OWN          2984
OTHER          96
NONE            3
Name: home_ownership, dtype: int64
CÁC GIÁ TRỊ UNIQUE CÓ TRONG verification_status là:  Not Verified       16696
Verified           12289
Source Verified     9722
Name: verification_status, dtype: int64
CÁC GIÁ TRỊ UNIQUE CÓ TRONG term là:   36 months    29040
 60 months     9667
Name: term, dtype: int64
CÁC GIÁ TRỊ UNIQUE CÓ TRONG addr_state là:  CA    6958
NY    3713
FL    2791
TX    2667
NJ    1798
IL    1483
PA    1473
VA    1376
GA    1364
MA    1301
OH    1179
MD    1026
AZ     850
WA     822
CO     770
NC     753
CT     730
MI     712
MO     670
MN     603
NV     481
SC     462
WI     441
AL     437
OR     436
LA     430
KY     315
OK     290
KS     260
UT     254
AR     237
DC     209
RI     196
NM     184
WV     172
NH     166
HI     166
DE     113
MT      83
WY      80
AK      78
SD      61
VT      53
MS      19
TN      17
IN       9
ID    

In [39]:
print(df["purpose"].value_counts())
print(df["title"].value_counts())

debt_consolidation    18130
credit_card            5039
other                  3864
home_improvement       2897
major_purchase         2154
small_business         1762
car                    1510
wedding                 929
medical                 680
moving                  576
vacation                375
house                   369
educational             320
renewable_energy        102
Name: purpose, dtype: int64
Debt Consolidation                                            2104
Debt Consolidation Loan                                       1632
Personal Loan                                                  642
Consolidation                                                  494
debt consolidation                                             485
Credit Card Consolidation                                      353
Home Improvement                                               346
Debt consolidation                                             324
Small Business Loan                         

Ta sẽ sửa lại định dạng ở một số cột có chứa % về dạng float

In [40]:
df = df.drop(['last_credit_pull_d', 'earliest_cr_line', 'addr_state', 'title'], axis=1)
df["int_rate"] = df["int_rate"].str.rstrip("%").astype("float")
df["revol_util"] = df["revol_util"].str.rstrip("%").astype("float")
df.head(5)

,loan_amnt,term,int_rate,installment,home_ownership,annual_inc,verification_status,loan_status,purpose,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc
0,5000.0,36 months,10.65,162.87,RENT,24000.0,Verified,1,credit_card,27.65,0.0,1.0,3.0,0.0,13648.0,83.7,9.0
1,2500.0,60 months,15.27,59.83,RENT,30000.0,Source Verified,0,car,1.00,0.0,5.0,3.0,0.0,1687.0,9.4,4.0
2,2400.0,36 months,15.96,84.33,RENT,12252.0,Not Verified,1,small_business,8.72,0.0,2.0,2.0,0.0,2956.0,98.5,10.0
3,10000.0,36 months,13.49,339.31,RENT,49200.0,Source Verified,1,other,20.00,0.0,1.0,10.0,0.0,5598.0,21.0,37.0
5,5000.0,36 months,7.90,156.46,RENT,36000.0,Source Verified,1,wedding,11.20,0.0,3.0,9.0,0.0,7963.0,28.3,12.0


Bấy giờ ta sẽ chuyển những cột có dạng category sang numerical bằng get_dummie method

In [45]:
cat_columns = ["home_ownership", "verification_status", "purpose", "term"]
dummies_df = pd.get_dummies(df[cat_columns])
df = pd.concat([df, dummies_df], axis=1)
df.drop(cat_columns, axis=1, inplace=True)
print(df.shape)
df.head(3)

(38707, 37)


,loan_amnt,int_rate,installment,annual_inc,loan_status,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,revol_bal,revol_util,total_acc,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OTHER,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified,purpose_car,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,term_ 36 months,term_ 60 months
0,5000.0,10.65,162.87,24000.0,1,27.65,0.0,1.0,3.0,0.0,13648.0,83.7,9.0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,2500.0,15.27,59.83,30000.0,0,1.00,0.0,5.0,3.0,0.0,1687.0,9.4,4.0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,2400.0,15.96,84.33,12252.0,1,8.72,0.0,2.0,2.0,0.0,2956.0,98.5,10.0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0


In [46]:
df.loan_status.value_counts()

1    33092
0     5615
Name: loan_status, dtype: int64

Ta nhận thấy cột target là loan_status có sự phân bổ giá trị 0 và 1 quá chênh lệch, điều này sẽ dẫn đến sự sai lệch trong cách tính accuracy của model nếu ta dùng pp thông thường như rmse, bởi lí do là vì cột target chứa quá nhiều giá trị 1, do đó khi ta làm predictive modelling thì hầu như kết quả đầu ra sẽ thường nghiêng về 1 nhiều hơn, do đó dẫn đến sai lệch trong dự báo, ta cần dùng các chỉ số khác đó là False Positive Rate ( phần trăm kết quả đầu ra cho 1 bị dự đoán sai) và True positive rate(Phần trăm đầu ra 1 dc dự đoán đúng)

In [48]:
target = df.loan_status
df = df.drop('loan_status', axis = 1)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict

lr = LogisticRegression()

predict = cross_val_predict(lr, df, target, cv=3)
predict

array([1, 1, 1, ..., 1, 1, 1])

In [57]:
False_positive = (predict == 1) & (target == 0)

True_postive = (predict == 1) & (target == 1)
False_negative = (predict == 0) & (target == 1)
True_negative = (predict == 0) & (target == 0)

True_positive_rate = len(predict[True_postive]) / (len(predict[True_postive]) + len(predict[False_negative]))
False_positive_rate = len(predict[False_positive]) / (len(predict[False_positive]) + len(predict[True_negative]))
print(True_positive_rate)
print(False_positive_rate)
                                                      
                                                      

0.9988214674241509
0.9966162065894925


2 chỉ số trên chứng tỏ mô hình đã dự đoán hầu như đầu ra đều là 1, dễ hiểu điều này vì 1 chiếm phần lớn trong target value ở tập dataset, có 2 cách để resovle cái imbalanced class này, cách 1 đó là ta training với tập dataset đã được chọn sao cho ở đó số lượng 1 và 0 là đều nhau, cách 2 gọi là penalize misclassfication

ta sẽ sử dụng cách 2 với thư viện scikit learn và set class_weight = imbalanced

In [58]:
lr = LogisticRegression(class_weight="balanced")
predict = cross_val_predict(lr, df, target, cv=3)

False_positive = (predict == 1) & (target == 0)

True_postive = (predict == 1) & (target == 1)
False_negative = (predict == 0) & (target == 1)
True_negative = (predict == 0) & (target == 0)

True_positive_rate = len(predict[True_postive]) / (len(predict[True_postive]) + len(predict[False_negative]))
False_positive_rate = len(predict[False_positive]) / (len(predict[False_positive]) + len(predict[True_negative]))
print(True_positive_rate)
print(False_positive_rate)

0.6677444699625287
0.39038290293855743


Ta thấy lúc này chỉ số đã trờ nên cân bằng hơn rất nhiều, so far so good 

Ta có thể tự mình penalize the weight bằng cách

In [59]:
penalty = {
            0: 10,
            1: 1
           }
lr = LogisticRegression(class_weight=penalty)
predict = cross_val_predict(lr, df, target, cv=3)

False_positive = (predict == 1) & (target == 0)

True_postive = (predict == 1) & (target == 1)
False_negative = (predict == 0) & (target == 1)
True_negative = (predict == 0) & (target == 0)

True_positive_rate = len(predict[True_postive]) / (len(predict[True_postive]) + len(predict[False_negative]))
False_positive_rate = len(predict[False_positive]) / (len(predict[False_positive]) + len(predict[True_negative]))
print(True_positive_rate)
print(False_positive_rate)


0.23277529312220477
0.08441674087266252


ta thấy tỉ lệ đã trờ nên tốt hơn
NHư vậy trên đây là 1 project đơn giản nhưng có nhiều ý nghĩa, nó giúp ta định hình được 1 mô hình machine learning sẽ hoạt động ra sao
cách handle imbalanced như thế nào